# 03 – TSDF fusion with Open3D (reference) and Marching Cubes (Milestones 3–4)

## The math

Each voxel $x$ (center $\mathbf{x}$) stores a truncated signed distance $D(x)$ and a weight $W(x)$.
For one depth frame with intrinsics $K$ and pose $T_\text{cam→world}$:

1. Transform the voxel center into the camera: $\mathbf{x}_c = T^{-1}_\text{cam→world}\,\mathbf{x}$, project it to pixel $(u,v) = \pi(K\mathbf{x}_c)$.
2. **Projective signed distance** along the ray: $\;\text{sdf} = d(u,v) - z(\mathbf{x}_c)$
   (positive: voxel is in front of the measured surface; negative: behind it).
3. **Truncate**: skip the voxel if $\text{sdf} < -\mu$ (too far behind the surface, probably occluded);
   otherwise $\;\text{tsdf} = \min(1,\ \text{sdf}/\mu)$.
4. **Weighted running average** (Curless & Levoy 1996; KinectFusion uses $w=1$):
$$D \leftarrow \frac{W D + w\,\text{tsdf}}{W + w}, \qquad W \leftarrow W + w$$

The surface is the zero level set $D = 0$, extracted as a triangle mesh with **Marching Cubes**.
Averaging many noisy observations cancels much of the per-frame noise, and floating points from
single frames get outvoted by the views that saw empty space there.

## What Open3D's implementation does / does not do

* `VoxelBlockGrid`: voxel blocks in a hash map (voxel hashing, Nießner et al. 2013), allocated **only
  near observed surfaces**. Free space far from surfaces is not stored, so it cannot be distinguished
  from unknown space. That is the main reason to write our own TSDF later.
* Constant weights $w = 1$; no space carving.

Code: `src/volumetric_reasoning/geometry/tsdf_open3d.py`, metrics in `geometry/mesh_eval.py`.

In [ ]:
%matplotlib inline
import time
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d

from volumetric_reasoning.config import load_config, resolve_path
from volumetric_reasoning.datasets import ScanNetPPIphoneScene
from volumetric_reasoning.geometry.tsdf_open3d import integrate_open3d, extract_mesh, voxel_stats
from volumetric_reasoning.geometry.mesh_eval import compare_meshes
from volumetric_reasoning.geometry.raycast import make_raycasting_scene, render_normals, render_topdown
from volumetric_reasoning.geometry.backprojection import frames_to_points

# ---- choose the scene here ---------------------------------------------
SCENE_ID = "1ada7a0617"   # None = use dataset.scene_id from configs/default.yaml / local.yaml
# --------------------------------------------------------------------------

cfg = load_config()
if SCENE_ID is not None:
    cfg["dataset"]["scene_id"] = SCENE_ID
tc, dc = cfg["tsdf"], cfg["depth"]
scene = ScanNetPPIphoneScene(resolve_path(cfg["dataset"]["root"]), cfg["dataset"]["scene_id"])
frames = scene.load_frames(scene.select_frames(stride=tc["frame_stride"]))
ref_mesh = o3d.io.read_triangle_mesh(str(scene.mesh_path)); ref_mesh.compute_vertex_normals()
out_dir = resolve_path(cfg["output"]["root"]) / scene.scene_id
out_dir.mkdir(parents=True, exist_ok=True)
print(f"{scene.scene_id}: {len(frames)} frames (every {tc['frame_stride']}th), TSDF settings {tc}")

## 1. Fuse and extract the mesh

In [ ]:
t0 = time.time()
vbg = integrate_open3d(frames, tc["voxel_size"], tc["trunc_voxels"], dc["min_depth"], dc["max_depth"])
mesh = extract_mesh(vbg, tc["weight_threshold"])
print(f"integration + Marching Cubes: {time.time() - t0:.1f} s")
print(f"mesh: {len(mesh.vertices):,} vertices, {len(mesh.triangles):,} triangles")
print("voxels:", voxel_stats(vbg))

metrics = compare_meshes(mesh, ref_mesh, tau=0.05)
print({k: round(v, 3) for k, v in metrics.items()})

path = out_dir / f"tsdf_open3d_{int(tc['voxel_size'] * 1000)}mm.ply"
o3d.io.write_triangle_mesh(str(path), mesh)
print("saved ->", path)

## 2. Same viewpoints: RGB vs. TSDF mesh vs. laser mesh

Shaded renderings (headlight shading) from the poses of a few input frames.

In [ ]:
rc_pred = make_raycasting_scene(mesh)
rc_ref = make_raycasting_scene(ref_mesh)
W, H = 512, 384
show = frames[:: max(1, len(frames) // 4)][:4]
fig, ax = plt.subplots(len(show), 3, figsize=(15, 3.7 * len(show)))
for row, f in zip(ax, show):
    K = f.K_depth * np.array([[W / 256], [H / 192], [1]])  # scale intrinsics to render size
    row[0].imshow(f.rgb); row[0].set_title(f"RGB frame {f.index}")
    row[1].imshow(render_normals(rc_pred, K, f.cam_to_world, W, H), cmap="gray", vmin=0, vmax=1); row[1].set_title("TSDF mesh (Open3D)")
    row[2].imshow(render_normals(rc_ref, K, f.cam_to_world, W, H), cmap="gray", vmin=0, vmax=1); row[2].set_title("laser mesh (reference)")
    for a in row: a.axis("off")
plt.tight_layout()

## 3. Top-down view with the ceiling removed

Rays start at height `Z_CUT` and go straight down, so we look into the room like a dollhouse.
Left: raw back-projected points below `Z_CUT` (highest point per cell), middle: TSDF mesh, right: laser mesh.

In [ ]:
Z_CUT = 2.0
verts = np.asarray(ref_mesh.vertices)
xy_min, xy_max = verts[:, :2].min(0) - 0.2, verts[:, :2].max(0) + 0.2

pts, _, _ = frames_to_points(frames[::3], dc["min_depth"], dc["max_depth"])
pts = pts[pts[:, 2] < Z_CUT]
res = 0.01
ix = ((pts[:, 0] - xy_min[0]) / res).astype(int); iy = ((xy_max[1] - pts[:, 1]) / res).astype(int)
H_img, W_img = int((xy_max[1] - xy_min[1]) / res) + 1, int((xy_max[0] - xy_min[0]) / res) + 1
keep = (ix >= 0) & (ix < W_img) & (iy >= 0) & (iy < H_img)
height = np.full((H_img, W_img), np.nan)
order = np.argsort(pts[keep, 2])  # later (higher) points overwrite lower ones
height[iy[keep][order], ix[keep][order]] = pts[keep, 2][order]

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ext = [xy_min[0], xy_max[0], xy_min[1], xy_max[1]]
ax[0].imshow(height, cmap="viridis", extent=ext); ax[0].set_title("raw points (color = height)")
ax[1].imshow(render_topdown(rc_pred, xy_min, xy_max, Z_CUT, res), cmap="gray", vmin=0, vmax=1, extent=ext); ax[1].set_title("TSDF mesh")
ax[2].imshow(render_topdown(rc_ref, xy_min, xy_max, Z_CUT, res), cmap="gray", vmin=0, vmax=1, extent=ext); ax[2].set_title("laser mesh")
for a in ax: a.set_xlabel("x [m]")
ax[0].set_ylabel("y [m]")
plt.tight_layout()

## 4. Where is the reconstruction wrong?

Distance of every TSDF mesh vertex to the laser mesh. A copy of the mesh colored by this error is saved for 3D viewing.

In [ ]:
d = rc_ref.compute_distance(o3d.core.Tensor(np.asarray(mesh.vertices, dtype=np.float32))).numpy()
print(f"vertex error: median {np.median(d)*100:.1f} cm, 90th pct {np.percentile(d, 90)*100:.1f} cm, > 5 cm: {(d > 0.05).mean()*100:.1f}%")
plt.figure(figsize=(6, 3)); plt.hist(np.minimum(d, 0.2) * 100, bins=80); plt.xlabel("distance to laser mesh [cm] (clipped at 20)"); plt.ylabel("vertices")

err_mesh = o3d.geometry.TriangleMesh(mesh)
err_mesh.vertex_colors = o3d.utility.Vector3dVector(plt.cm.magma(np.minimum(d / 0.1, 1.0))[:, :3])
o3d.io.write_triangle_mesh(str(out_dir / "tsdf_open3d_error.ply"), err_mesh)
print("saved error-colored mesh ->", out_dir / "tsdf_open3d_error.ply")

## 5. Voxel size: accuracy vs. number of voxels

Re-integrates the same frames at several resolutions. *Observed voxels* = voxels with weight > 0 (Open3D only
stores voxels near surfaces, so this is **not** the number of voxels in the room; a dense grid over the room's
bounding box is listed for comparison). *Near-surface voxels* = within the truncation band.

In [ ]:
VOXEL_SIZES = [0.01, 0.02, 0.04, 0.08]
rows, tops = [], []
bbox_volume = np.prod(verts.max(0) - verts.min(0))
for vs in VOXEL_SIZES:
    t0 = time.time()
    v = integrate_open3d(frames, vs, tc["trunc_voxels"], dc["min_depth"], dc["max_depth"], with_color=False)
    m = extract_mesh(v, tc["weight_threshold"])
    r = compare_meshes(m, ref_mesh)
    st = voxel_stats(v)
    rows.append(dict(voxel_cm=vs * 100, seconds=time.time() - t0, triangles=len(m.triangles),
                     dense_bbox_voxels=int(bbox_volume / vs**3), **st, **r))
    tops.append(render_topdown(make_raycasting_scene(m), xy_min, xy_max, Z_CUT, 0.01))

print(f"{'voxel':>6} {'time':>6} {'triangles':>10} {'dense bbox':>11} {'observed':>11} {'near-surf':>11} {'acc':>6} {'comp':>6} {'F@5cm':>6}")
for r in rows:
    print(f"{r['voxel_cm']:4.0f}cm {r['seconds']:5.1f}s {r['triangles']:>10,} {r['dense_bbox_voxels']:>11,} {r['observed_voxels']:>11,} "
          f"{r['near_surface_voxels']:>11,} {r['accuracy_median_m']*100:5.1f}c {r['completeness_median_m']*100:5.1f}c {r['fscore']:6.3f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
x = [r["voxel_cm"] for r in rows]
ax[0].plot(x, [r["fscore"] for r in rows], "o-", label="F-score @ 5 cm")
ax[0].plot(x, [r["precision"] for r in rows], "s--", label="precision")
ax[0].plot(x, [r["recall"] for r in rows], "^--", label="recall")
ax[0].set_xlabel("voxel size [cm]"); ax[0].legend(); ax[0].set_title("reconstruction quality")
ax[1].loglog(x, [r["dense_bbox_voxels"] for r in rows], "o-", label="dense grid over room bbox")
ax[1].loglog(x, [r["observed_voxels"] for r in rows], "s-", label="observed (Open3D)")
ax[1].loglog(x, [r["near_surface_voxels"] for r in rows], "^-", label="near-surface")
ax[1].set_xlabel("voxel size [cm]"); ax[1].set_ylabel("voxels"); ax[1].legend(); ax[1].set_title("voxel counts")
plt.tight_layout()

fig, ax = plt.subplots(1, len(tops), figsize=(5 * len(tops), 4.5))
for a, img, vs in zip(ax, tops, VOXEL_SIZES):
    a.imshow(img, cmap="gray", vmin=0, vmax=1, extent=ext); a.set_title(f"{vs*100:.0f} cm voxels"); a.axis("off")
plt.tight_layout()

## 6. Optional: interactive 3D view

`MAX_HEIGHT` removes the ceiling. `MESH = mesh` shows RGB colors, `MESH = err_mesh` shows the error coloring
(black = correct, yellow = ≥ 10 cm off). Back faces are hidden, so walls facing away from you disappear.

In [ ]:
SHOW_3D = False
MAX_HEIGHT = 2.0
MESH = mesh
if SHOW_3D:
    view = MESH
    if MAX_HEIGHT is not None:
        lo, hi = view.get_min_bound(), view.get_max_bound()
        view = view.crop(o3d.geometry.AxisAlignedBoundingBox(lo, [hi[0], hi[1], MAX_HEIGHT]))
    o3d.visualization.draw_geometries([view], front=[0.0, -0.4, 1.0], up=[0, 0, 1], lookat=view.get_center(), zoom=0.6)

## Findings

- Fill in after running.
- Next: our own dense TSDF (with free / unknown space), validated against this Open3D result.